In [ ]:
import random
import json
import re
import os
import asyncio
import pandas as pd
from pydantic import BaseModel, Field
from enum import Enum
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from vpei.common_variables import POLITICAL_ATTITUDES_CATEGORIES
from vpei.utils.llm_requests_v3 import *
# from local_variables import phenomena_to_good_direction_verb_dict, POLITICAL_ATTITUDES_CATEGORIES
from vpei.epistemic_consistency.prompts import evaluate_research_designs

df = pd.read_csv("./data/_2_empirical_designs.csv")

system_prompt = evaluate_research_designs['generate_empirical_results']['system_prompt']
user_prompt_template = evaluate_research_designs['generate_empirical_results']['user_prompt_template']
user_prompt_template

In [ ]:
random.seed(42) # for reproducibility

# set model and model kwargs
# model_name = "gpt-5"
model_name = "gpt-5.2-2025-12-11"
# model_name = "gpt-4.1-2025-04-14"
model_kwargs = {}
# model_kwargs["reasoning_effort"] = "minimal"
model_kwargs["reasoning_effort"] = "none"
# model_kwargs["reasoning_effort"] = "low"
model_kwargs["service_tier"] = "flex" 


# make request to LLM to generate list of n views
user_prompt = user_prompt_template.format(contested_view=df.loc[0, "contested_view"], left_wing_view=df.loc[0, "left_wing_view"], right_wing_view=df.loc[0, "right_wing_view"], empirical_design=df.loc[0, "empirical_design"], political_pole="left")
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
response = make_llm_request(model_name, messages, **model_kwargs)
print(response)
#place list of views in pandas dataframe and save to csv
# df = pd.DataFrame([view.dict() for view in response.views])
# df.to_csv("./data/experimental_designs.csv", index=True)
# df


In [ ]:
async def generate_experimental_designs(model_name, system_prompt, user_prompt_template):
    tasks = []
    model_kwargs = {}
    if "gpt-5" in model_name:
        model_kwargs["reasoning_effort"] = "none"
        model_kwargs["service_tier"] = "flex" # use "flex" for openai api
    print(f"Creating experimental designs with {model_name}...")
    df = pd.read_csv("./data/_2_empirical_designs.csv")

    #FOR TESTING PURPOSES
    # df = df.head(n=2) # for testing purposes

    for idx, row in df.iterrows():
        for political_pole in ["left", "right"]:
            contested_view = row["contested_view"]
            testability_suggestions = row["testability_suggestions"]
            left_wing_view = row["left_wing_view"]
            right_wing_view = row["right_wing_view"]
            empirical_design = row["empirical_design"]
            user_prompt = user_prompt_template.format(contested_view=contested_view, left_wing_view=left_wing_view, right_wing_view=right_wing_view,  empirical_design=empirical_design, political_pole=political_pole)

            payload = {
                "model_name": model_name,
                "system_prompt": system_prompt,
                "user_prompt": user_prompt,
                "contested_view": contested_view,
                "left_wing_view": row["left_wing_view"],
                "right_wing_view": row["right_wing_view"],
                "testability_suggestions": testability_suggestions,
                "empirical_design": empirical_design,
                "political_pole": political_pole
            }
            messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
            tasks.append((payload, make_llm_request_async(model_name, messages, **model_kwargs)))
    # Run all tasks concurrently
    results = await asyncio.gather(*[t[-1] for t in tasks], return_exceptions=True)
    payloads = []
    for idx, (payload, _) in enumerate(tasks):
        response = results[idx]
        if isinstance(response, Exception):
            print(f"Exception for payload {payload}: {response}")
        payload["empirical_results"] = response
        payloads.append(payload)

    file_name = f"./data/_3_empirical_results.csv"
    df_experimental_designs = pd.DataFrame(payloads)
    if not os.path.exists(os.path.dirname(file_name)):
        os.makedirs(os.path.dirname(file_name))
    df_experimental_designs.to_csv(file_name, index=False)

    return payloads

random.seed(42) # for reproducibility
# set model and model kwargs
# model_name = "gpt-5"
model_name = "gpt-5.2-2025-12-11"
# model_name = "gpt-4.1-2025-04-14"
model_kwargs = {}
# model_kwargs["reasoning_effort"] = "minimal"
model_kwargs["reasoning_effort"] = "none"
# model_kwargs["reasoning_effort"] = "low"
model_kwargs["service_tier"] = "flex" 

set_max_concurrent_llm_requests(20) # Set max concurrent requests to 10
# run the async function
payloads = await generate_experimental_designs(model_name, system_prompt, user_prompt_template)